# 04 · Feature engineering  *(sección 4 del script)*

**Qué hacemos:** construimos **9 variables nuevas** que no estaban en el dataset original (`items_por_pedido` ya se calculó en la limpieza; las otras 8 se crean acá):

| Variable | Qué mide |
|---|---|
| `tiempo_entrega_dias` | días entre la compra y la entrega real |
| `dias_vs_estimado` | días de diferencia entre entrega real y fecha estimada (positivo = tarde) |
| `envio_demorado` | binaria 1/0: ¿se entregó después de la fecha estimada? |
| `flete_ratio` | qué tan grande es el flete respecto del precio del producto |
| `reseña_positiva` | review_score binarizado (1 = 4-5 ★, 0 = 1-2 ★, NaN = 3 ★) |
| `region_sudeste` | Sudeste (SP/RJ/MG/ES) vs. resto del país |
| `mes_compra` | mes de la compra (para estacionalidad) |
| `distancia_km` | distancia geográfica cliente-vendedor (Haversine) |
| `volumen_cm3` | largo × alto × ancho del producto |

**Para qué:** son las variables con las que se contestan todas las preguntas del análisis: tiempos, demoras, costos, satisfacción, geografía y logística. Sin ellas, el análisis se quedaría en describir columnas crudas.

## Carga del insumo (celda estándar)

**Qué hacemos:** cargamos `01_df_limpio.csv` (notebook 03), rehidratamos los tipos `datetime` (el CSV no los conserva) y preparamos `mes_compra`. Si falta el archivo, el error indica qué notebook correr.

**Para qué:** para partir siempre de la misma base limpia y con los tipos correctos.

In [1]:
# Celda estándar: imports + rutas + paleta (explicadas en 01_configuracion_inicial)
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")

BASE = Path.cwd()
if not (BASE / "data").exists() and (BASE.parent / "data").exists():
    BASE = BASE.parent

PIPELINE_DIR = BASE / "data" / "pipeline"
SUDESTE = {"SP", "RJ", "MG", "ES"}
pd.set_option("display.max_columns", 50)

COLUMNAS_FECHA = [
    "shipping_limit_date", "order_purchase_timestamp", "order_approved_at",
    "order_delivered_carrier_date", "order_delivered_customer_date",
    "order_estimated_delivery_date",
]

RUTA_LIMPIO = PIPELINE_DIR / "01_df_limpio.csv"
if not RUTA_LIMPIO.exists():
    raise FileNotFoundError(
        f"No existe {RUTA_LIMPIO}. Ejecutá primero el notebook 03_carga_y_limpieza.ipynb."
    )

df = pd.read_csv(RUTA_LIMPIO)
for col in COLUMNAS_FECHA:
    df[col] = pd.to_datetime(df[col], errors="coerce")

print(f"filas={df.shape[0]:,} columnas={df.shape[1]}")

filas=112,650 columnas=34


## 1 y 2 — Tiempos de entrega: `tiempo_entrega_dias` y `dias_vs_estimado`

**Qué hacemos:** restamos fechas y pasamos a días (`.dt.days`):

- `tiempo_entrega_dias` = `entrega_cliente − compra` → cuánto tardó **en la realidad**.
- `dias_vs_estimado` = `entrega_cliente − fecha_estimada` → cuánto se **desvió** de lo prometido. **Positivo = llegó tarde; negativo = llegó antes.**

**Para qué:** son dos preguntas distintas del negocio (¿cuánto tarda? vs. ¿cumple lo que prometió?) y de ahí sale la variable binaria de la consigna (`envio_demorado`).

In [2]:
df["tiempo_entrega_dias"] = (
    df["order_delivered_customer_date"] - df["order_purchase_timestamp"]
).dt.days

df["dias_vs_estimado"] = (
    df["order_delivered_customer_date"] - df["order_estimated_delivery_date"]
).dt.days

print(df[["tiempo_entrega_dias", "dias_vs_estimado"]].describe().round(1).to_string())
print()
print(f"Pedidos sin fecha de entrega real (no entregados): "
      f"{df['order_delivered_customer_date'].isna().sum():,} -> ambas variables en NaN")

       tiempo_entrega_dias  dias_vs_estimado
count             110196.0          110196.0
mean                  12.0             -12.0
std                    9.5              10.2
min                    0.0            -147.0
25%                    6.0             -17.0
50%                   10.0             -13.0
75%                   15.0              -7.0
max                  209.0             188.0

Pedidos sin fecha de entrega real (no entregados): 2,454 -> ambas variables en NaN


**Insight:** la media de `dias_vs_estimado` es **negativa**: en promedio Olist entrega *antes* de la fecha estimada — las estimaciones tienen un margen conservador. Por eso, cuando ese margen no alcanza, el atraso resultante "golpea" doble (se verá en la satisfacción, notebook 10). Los pedidos no entregados quedan en `NaN`: no se les inventa fecha.

## 3 — `envio_demorado` (la variable binaria de la consigna)

**Qué hacemos:** `1` si la entrega real fue **posterior** a la estimada, `0` si llegó a tiempo… y **`NaN` si el pedido nunca se entregó** (`np.where` con la condición de nulos).

**Para qué:** es la variable objetivo del análisis de demoras (y de los modelos del script 2). El `NaN` en los no entregados es deliberado: un pedido cancelado o en camino **no es "a tiempo" ni "demorado"** — no tiene entrega que comparar; forzarlo a 0 inflaría artificialmente el % de entregas puntuales.

In [3]:
df["envio_demorado"] = np.where(
    df["order_delivered_customer_date"].isna(),
    np.nan,
    (df["order_delivered_customer_date"] > df["order_estimated_delivery_date"]).astype(float),
)

print(df["envio_demorado"].value_counts(dropna=False).sort_index().to_string())
print()
print(f"% envío demorado (sobre entregados): {df['envio_demorado'].mean(skipna=True) * 100:.1f}%")

envio_demorado
0.0    101481
1.0      8715
NaN      2454

% envío demorado (sobre entregados): 7.9%


**Insight:** casi **1 de cada 10 pedidos entregados** llega después de la fecha estimada. Es la primera señal de que la logística es un problema relevante, mucho antes de llegar a los tests formales del notebook 10.

## 4 — `flete_ratio` (flete relativo al precio)

**Qué hacemos:** dividimos `freight_value / price`, reemplazando previamente los precios iguales a 0 por `NaN` (para no dividir por cero).

**Para qué:** el valor absoluto del flete no dice nada sin saber el valor del producto: R$ 20 de flete es caro para un producto de R$ 30 y trivial para uno de R$ 500. El ratio mide **qué proporción del precio del producto representa llevarlo** — clave en un marketplace que envía a todo Brasil.

In [4]:
df["flete_ratio"] = df["freight_value"] / df["price"].replace(0, np.nan)

print(df["flete_ratio"].describe().round(3).to_string())
print()
print(f"Filas con price = 0 (ratio quedó en NaN): {(df['price'] == 0).sum()}")

count    112650.000
mean          0.321
std           0.350
min           0.000
25%           0.134
50%           0.231
75%           0.393
max          26.235

Filas con price = 0 (ratio quedó en NaN): 0


## 5 — `reseña_positiva` (review_score binarizado)

**Qué hacemos:** función `clasificar_reseña`: **1** si el puntaje es 4-5 ★, **0** si es 1-2 ★, y **`NaN` si es 3 ★** (o si no hay reseña).

**Para qué:** convierte la satisfacción en una variable de clasificación para los modelos. El puntaje 3 se deja **fuera** a propósito: es "neutral/indiferente", meterlo de un lado o del otro distorsionaría las clases (la distribución de scores, en el notebook 06, muestra que hay muy pocos 3 igualmente).

In [5]:
def clasificar_reseña(score):
    if pd.isna(score):
        return np.nan
    if score <= 2:
        return 0
    if score >= 4:
        return 1
    return np.nan      # score == 3 -> neutral, queda fuera de la clase positiva/negativa

df["reseña_positiva"] = df["review_score"].apply(clasificar_reseña)

print(df["reseña_positiva"].value_counts(dropna=False).sort_index().to_string())
print()
print(f"% de reseñas positivas (sobre las no neutrales): "
      f"{df['reseña_positiva'].mean(skipna=True) * 100:.1f}%")

reseña_positiva
0.0    17989
1.0    84343
NaN    10318

% de reseñas positivas (sobre las no neutrales): 82.4%


## 6 — `region_sudeste` y `mes_compra`

**Qué hacemos:**

- `region_sudeste`: mapea `customer_state ∈ {SP, RJ, MG, ES}` → `"Sudeste"`, resto → `"Resto del país"`.
- `mes_compra`: normaliza la fecha de compra al primer día del mes (para agrupar la estacionalidad mes a mes).

**Para qué:** para contrastar la hipótesis 3 (la brecha geográfica) con una partición simple pero significativa — el eje Sudeste concentra clientes *y* vendedores — y para graficar la serie de ventas mensuales.

In [6]:
df["region_sudeste"] = df["customer_state"].isin(SUDESTE).map(
    {True: "Sudeste", False: "Resto del país"}
)
df["mes_compra"] = df["order_purchase_timestamp"].dt.to_period("M").dt.to_timestamp()

print(df["region_sudeste"].value_counts().to_string())
print()
print(f"Meses de compra distintos: {df['mes_compra'].nunique()}")
print(f"Top 5 estados: {', '.join(df['customer_state'].value_counts().head(5).index)}")

region_sudeste
Sudeste           77413
Resto del país    35237

Meses de compra distintos: 24
Top 5 estados: SP, RJ, MG, RS, PR


**Insight:** el Sudeste domina en cantidad de pedidos (coherente con la concentración de población y de vendedores), pero el "Resto del país" también pesa — es decir, hay una porción grande de clientes que queda **fuera del eje logístico principal**: justo el grupo comparado en la hipótesis 3.

## 7 — `distancia_km` (fórmula de Haversine)

**Qué hacemos:** calculamos la distancia del **arco más corto entre dos puntos sobre la esfera terrestre**, a partir de las coordenadas del cliente y del vendedor:

$$d = 2R \cdot \arcsin\left(\sqrt{\sin^2\left(\frac{\Delta lat}{2}\right) + \cos(lat_1)\,\cos(lat_2)\,\sin^2\left(\frac{\Delta lon}{2}\right)}\right)$$

Pasos del código: convertir grados → **radianes**, restar latitudes/longitudes, aplicar la fórmula con $R = 6371$ km (radio de la Tierra).

**Para qué:** es la medida **continua** de lejanía (en kilómetros, no en categorías) para responder si la distancia real cliente-vendedor influye en el tiempo de entrega y en el flete — complementando la comparación gruesa Sudeste/resto.

In [7]:
def haversine_km(lat1, lon1, lat2, lon2):
    R = 6371.0  # radio de la Tierra en km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat / 2) ** 2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon / 2) ** 2
    return 2 * R * np.arcsin(np.sqrt(a))

df["distancia_km"] = haversine_km(
    df["customer_lat"], df["customer_lng"], df["seller_lat"], df["seller_lng"]
)

print(df["distancia_km"].describe().round(1).to_string())
print()
print(f"Sin distancia calculable (faltan coordenadas): {df['distancia_km'].isna().sum():,} filas")

count    112087.0
mean        596.7
std         588.8
min           0.0
25%         184.0
50%         431.6
75%         792.2
max        3927.4

Sin distancia calculable (faltan coordenadas): 563 filas


**Insight:** la mediana de la distancia cliente-vendedor es de **varios cientos de kilómetros** — consistente con un país del tamaño de Brasil y con vendedores concentrados en el Sudeste. Ya se anticipa que la distancia va a pesar en los tiempos de entrega (se confirma con Spearman en el notebook 10). Los faltantes corresponden a los `NaN` de coordenadas validados en la limpieza.

## 8 — `volumen_cm3`

**Qué hacemos:** producto de las tres dimensiones del catálogo (`largo × alto × ancho`, en cm → resultado en cm³).

**Para qué:** para medir si el **tamaño físico** del producto influye en la logística (tiempo de entrega) y en el costo (`flete_ratio`) — junto a `product_weight_g`, que ya venía en el dataset.

In [8]:
df["volumen_cm3"] = df["product_length_cm"] * df["product_height_cm"] * df["product_width_cm"]

print(df["volumen_cm3"].describe().round(0).to_string())
print()
print(f"Dimensiones faltantes -> volumen en NaN: {df['volumen_cm3'].isna().sum():,} filas")

count    112632.0
mean      15244.0
std       23419.0
min         168.0
25%        2852.0
50%        6480.0
75%       18375.0
max      296208.0

Dimensiones faltantes -> volumen en NaN: 18 filas


### Verificación final de las 9 variables

In [9]:
print(f"% envío demorado (entregados): {df['envio_demorado'].mean(skipna=True) * 100:.1f}%")
print(f"Distancia cliente-vendedor: media={df['distancia_km'].mean():.0f} km, "
      f"mediana={df['distancia_km'].median():.0f} km")

NUEVAS = ["items_por_pedido", "tiempo_entrega_dias", "dias_vs_estimado", "envio_demorado",
          "flete_ratio", "reseña_positiva", "region_sudeste", "mes_compra",
          "distancia_km", "volumen_cm3"]
print()
print(f"Variables del análisis listas ({len(NUEVAS)}):")
for v in NUEVAS:
    tipo = str(df[v].dtype)
    print(f"  - {v:22s} {tipo:16s} nulos={df[v].isna().sum():,}")

% envío demorado (entregados): 7.9%
Distancia cliente-vendedor: media=597 km, mediana=432 km

Variables del análisis listas (10):
  - items_por_pedido       int64            nulos=0
  - tiempo_entrega_dias    float64          nulos=2,454
  - dias_vs_estimado       float64          nulos=2,454
  - envio_demorado         float64          nulos=2,454
  - flete_ratio            float64          nulos=0
  - reseña_positiva        float64          nulos=10,318
  - region_sudeste         object           nulos=0
  - mes_compra             datetime64[ns]   nulos=0
  - distancia_km           float64          nulos=563
  - volumen_cm3            float64          nulos=18


**Insight (síntesis):** las 9 variables quedaron construidas con nulos solo donde **corresponden** (pedidos no entregados, coordenadas ausentes, reseñas neutrales). Casi 1 de cada 10 pedidos entregados llega tarde y la distancia mediana cliente-vendedor es de cientos de km: con esto alcanza para arrancar el análisis.

## Guardar el artefacto del pipeline

**Qué hacemos:** escribimos `data/pipeline/02_df_features.csv` — la tabla "lista para analizar".

**Para qué:** todos los notebooks de análisis (05 → 10) cargan **este** archivo, así el pipeline no repite limpieza ni features en cada paso.

In [10]:
ruta_features = PIPELINE_DIR / "02_df_features.csv"
df.to_csv(ruta_features, index=False)
print(f"Artefacto del notebook 04 guardado -> {ruta_features}")
print(f"({df.shape[0]:,} filas x {df.shape[1]} columnas)")

Artefacto del notebook 04 guardado -> D:\Ciencia de Datos\Proyecto Integrador\trabajo_ integrador_versionNati\trabajo_ integrador_versionNati\data\pipeline\02_df_features.csv
(112,650 filas x 43 columnas)


**Siguiente paso:** `05_estadisticas_descriptivas.ipynb` — primera mirada numérica del dataset: conteos con % y estadísticos de las variables clave.